<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Conditional Routing & Qualitäts-Gate
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** Conditional Edges sind **Planen** in Reinform — der Meeting- & Briefing-Agent wählt zur Laufzeit den nächsten Schritt anhand des States, statt einem festen Pfad zu folgen. Das Qualitäts-Gate (Nachbesserungsschleife) ist der erste konkrete Baustein von **Prüfen**: Antworten werden bewertet und bei Bedarf nachgebessert, statt ungeprüft durchzureichen.



**Praxisanker:** Mara Vogt erhält je nach Fragetyp einen passenden Pfad: Fakten klären, Quellen vergleichen, Briefing zusammenfassen oder kontrolliert stoppen.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith-Variablen VOR allen LangChain-Imports setzen.
# Globales Tracing bleibt aus; ausgewählte Referenzläufe aktivieren es gezielt.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m09-conditional-routing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    setup_api_keys,
    mprint,
    mermaid,
    load_prompt,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)


# 1 | Übersicht
---

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


***StateGraph Basics*** zeigte sequentielle Nodes - jeder Node wird genau einmal durchlaufen.  
**Dieses Modul** erweitert das um **bedingte Verzweigungen**: Ein Node entscheidet zur Laufzeit, welchen Pfad der Graph nimmt.

**Was ist Conditional Routing?**

Eine **Routing-Funktion** liest den aktuellen State und gibt einen String zurück - den Namen des nächsten Nodes oder `END`. LangGraph folgt diesem Pfad.

```python
def mein_router(state: MeinState) -> str:
    if state["ergebnis"] == "ja":
        return "node_a"
    return "node_b"
```

**Drei Einsatzszenarien**

| Szenario | Routing-Kriterium | Beispiel |
|----------|------------------|----------|
| **Klassifikation** | LLM-Output | Briefing-Frage -> Definition / Retrieval / außerhalb des Korpus |
| **Tool-Loop** | `tool_calls` vorhanden? | Agent -> Tool -> Agent -> END |
| **Qualitäts-Gate** | Schwellenwert | Score < 0.7 -> Nachbessern -> Prüfen |


In [ ]:
#@markdown   <p><font size="4" color='green'>   Sequentiell vs. Konditional</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph SEQ["Sequentiell"]
        direction LR
        S1([START]) --> A[Node A] --> B[Node B] --> E1([END])
    end

    subgraph CON["Konditional"]
        direction LR
        S2([START]) --> R["🔀 Router"]
        R -->|Pfad 1| P["✅ Node Positiv"]
        R -->|Pfad 2| N["❌ Node Negativ"]
        R -->|Pfad 3| U["⚪ Node Neutral"]
        P & N & U --> E2([END])
    end
'''

mermaid(diagram, width=850)

**Wann Nodes + Edges, wann `bind_tools`?**

Die eine Frage, die alles klärt:

> *Gibt es mehrere Schritte oder Entscheidungen — oder brauche ich einmalig ein Ergebnis in einem bestimmten Format?*

| Situation | Mittel | Abschnitt |
|-----------|--------|-----------|
| Mehrere Schritte, Verzweigungen, Schleifen | Nodes + Edges | Kap. 3–4 |
| LLM soll *selbst* entscheiden ob Tools nötig | `bind_tools` (ohne Zwang) + `ToolNode` im Graph | – |
| Strukturiertes Ergebnis garantiert erzwingen | `bind_tools(tool_choice="required")` oder `with_structured_output` | Kap. 4 |

**Merksatz:**

> `bind_tools` = *„LLM, hier sind deine Werkzeuge"* (Bekanntmachung)  
> `ToolNode` im Graph = *„Jemand führt die Werkzeuge aus"* (Ausführung)  
> Nodes + Edges = *„Das ist der Plan, wer wann dran ist"* (Ablaufsteuerung)

Ein typischer Agenten-Loop braucht **alle drei** zusammen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Wann was? — Entscheidungsbaum</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    FRAGE{"Komplexität des\nWorkflows?"}

    FRAGE -->|"✅ Hoch\nVerzweigungen/Loops"| NEA["🔀 Nodes + Edges\n(Kap. 3–4)"]
    FRAGE -->|"🤖 Autonom\nLLM wählt, ToolNode führt aus"| BT["🔧 bind_tools + ToolNode"]
    FRAGE -->|"📦 Simpel\nStrukturiertes Ergebnis"| SO["🎯 with_structured_output\n(Kap. 4)"]

    NEA --> LOOP(("🔁"))
    BT  --> LOOP

    LOOP --- AGENT["<b>Agenten-Loop</b>\nKombiniert Steuerung,\nBekanntmachung & Ausführung"]

    SO --> END([Ziel erreicht])
    AGENT --> END

    %% Styles
    style FRAGE fill:#FF9800,stroke:#333,color:#fff
    style NEA   fill:#4CAF50,stroke:#333,color:#fff
    style BT    fill:#2196F3,stroke:#333,color:#fff
    style SO    fill:#9C27B0,stroke:#333,color:#fff
    style AGENT fill:#F5F5F5,stroke:#ccc,color:#333
    style LOOP  fill:#607D8B,stroke:#333,color:#fff
'''

mermaid(diagram, width=600)

# 2 | Conditional Edges
---

`add_edge()` verbindet immer fest. `add_conditional_edges()` verbindet dynamisch – basierend auf einer Funktion.

```python
**Feste Kante — Ablauf immer gleich**
builder.add_edge("node_a", "node_b")

**Bedingte Kante — Ablauf hängt vom State ab**
builder.add_conditional_edges(
    "node_a",         # Quell-Node
    mein_router,       # Routing-Funktion: State → str
    {                  # Explizite Path-Map für korrekte Graph-Visualisierung
        "pfad_1": "node_b",
        "pfad_2": "node_c",
        END:      END,
    }
)
```

Im Kurs wird die **Path-Map explizit angegeben**. So bleiben alle möglichen Router-Ausgänge und Kanten in `get_graph()` eindeutig sichtbar.

**State-Feld als Routing-Träger**

Bewährt: Die Routing-Entscheidung **im State speichern**, damit sie im Trace sichtbar ist:

```python
class MeinState(TypedDict):
    messages: Annotated[list, add_messages]
    routing:  str   # z.B. 'positiv' | 'negativ' | 'neutral'
```

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# State für Briefing-Routing
class RoutingState(TypedDict):
    messages: Annotated[list, add_messages]
    query: str   # Eingabe oder Briefing-Frage
    category: str   # 'definition' | 'retrieval' | 'out_of_corpus'
    answer:  str   # Finale Antwort

llm = init_chat_model(ROUTER)


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

# 3 | Routing-Funktion
---

Das Muster: Ein **Analyse-Node** schreibt seine Entscheidung ins State-Feld `category`.  
Die **Routing-Funktion** liest dieses Feld und leitet den Graphen weiter.

**Briefing-Beispiel:** Eine Frage wird nach Bearbeitungspfad klassifiziert und an den passenden Antwort-Node weitergeleitet.

> **Tipp:** `Literal[...]` als Rückgabetyp macht die möglichen Pfade explizit und verhindert Tippfehler.


**Warum ist die Routing-Funktion eine Edge und kein Node?**

In LangGraph übernehmen Funktionen zwei verschiedene Rollen - erkennbar an Rückgabetyp und Registrierung:

| Rolle | Aufgaben | Rückgabe | Registrierung |
|-------|---------|----------|---------------|
| **Node-Funktion** | Verarbeitung: LLM aufrufen, State schreiben | `dict` (State-Update) | `builder.add_node("name", funktion)` |
| **Edge-Funktion (Router)** | Entscheidung: welcher Node kommt als nächstes? | `str` (Node-Name) | `builder.add_conditional_edges("node", funktion)` |

```python
**Node: verarbeitet, gibt State-Update zurück**
def analyse_node(state: RoutingState) -> dict:
    return {"category": "retrieval"}

**Edge: liest State, gibt Pfad zurück**
def question_type_router(state: RoutingState) -> Literal["definition", "retrieval", "out_of_corpus"]:
    return state["category"]
```

> Die Routing-Funktion verändert den State nicht - sie liest ihn nur und leitet den Graphen weiter.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Briefing-Routing</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    START([START]) --> ANALYSE["Analyse-Node
Fragetyp erkennen"]
    ANALYSE -->|definition| DEF["Definition-Node
kurze Erklärung"]
    ANALYSE -->|retrieval| RET["Retrieval-Node
Dokumentenbezug prüfen"]
    ANALYSE -->|out_of_corpus| OOC["Out-of-Corpus-Node
Grenze markieren"]
    DEF & RET & OOC --> END([END])

    style ANALYSE fill:#FF9800,color:#fff
    style DEF     fill:#4CAF50,color:#fff
    style RET     fill:#2196F3,color:#fff
    style OOC     fill:#9E9E9E,color:#fff
'''

mermaid(diagram, width=650)


In [ ]:
# Prompt aus GitHub (mode='T': system+user Template mit {text})
briefing_routing_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m09_research_routing_prompt.md",
    mode="T")

def analyse_node(state: RoutingState) -> dict:
    """LLM klassifiziert die Frage anhand des zentralen Routing-Prompts."""
    prompt_value = briefing_routing_prompt.invoke({"text": state["query"]})
    response = llm.invoke(prompt_value.messages)
    inhalt = response.content.strip().lower()
    if "retrieval" in inhalt:
        routing = "retrieval"
    elif "out" in inhalt or "corpus" in inhalt:
        routing = "out_of_corpus"
    else:
        routing = "definition"
    return {"messages": [response], "category": routing}

def definition_node(state: RoutingState) -> dict:
    return {"answer": "Definitionspfad: Begriff kurz erklären; bei Bedarf auf spätere Dokumentenprüfung verweisen."}

def retrieval_node(state: RoutingState) -> dict:
    return {"answer": "Retrieval-Pfad: Dokumentenpassung prüfen und Antwort mit Quellenhinweis vorbereiten."}

def out_of_corpus_node(state: RoutingState) -> dict:
    return {"answer": "Nicht in den verfügbaren Dokumenten: Frage abgrenzen und keine unbelegte Fachantwort erfinden."}

def question_type_router(state: RoutingState) -> Literal["definition", "retrieval", "out_of_corpus"]:
    """Routing-Funktion: liest state['category'] und gibt Node-Namen zurück."""
    return state["category"]

briefing_builder = StateGraph(RoutingState)
briefing_builder.add_node("analyse", analyse_node)
briefing_builder.add_node("definition", definition_node)
briefing_builder.add_node("retrieval", retrieval_node)
briefing_builder.add_node("out_of_corpus", out_of_corpus_node)

briefing_builder.add_edge(START, "analyse")
briefing_builder.add_conditional_edges(
    "analyse",
    question_type_router,
    {
        "definition": "definition",
        "retrieval": "retrieval",
        "out_of_corpus": "out_of_corpus",
    },
)
briefing_builder.add_edge("definition", END)
briefing_builder.add_edge("retrieval", END)
briefing_builder.add_edge("out_of_corpus", END)

briefing_routing_agent = briefing_builder.compile()


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_conditional_edges(...)` — legt bedingte Übergänge zwischen Knoten fest
4. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
from IPython.display import Image as IPImage
display(IPImage(briefing_routing_agent.get_graph().draw_mermaid_png()))

In [ ]:
# Tests mit drei erwarteten Routing-Pfaden
from langsmith import tracing_context

test_fragen = [
    "Was bedeutet Retrieval Augmented Generation?",
    "Warum verbessert RAG die Zuverlässigkeit eines Meeting-Briefing-Agenten?",
    "Wie backe ich einen Apfelkuchen?",
]
erwartete_routen = ["definition", "retrieval", "out_of_corpus"]
tatsächliche_routen = []
run_cfg = {
    "run_name": "m09_kap3_briefing_routing",
    "tags": ["m09", "conditional", "research"],
}

# Nur dieser ausgewählte Referenzlauf wird an LangSmith übertragen.
with tracing_context(enabled=True):
    for _i, (text) in enumerate(test_fragen, 1):
        result = briefing_routing_agent.invoke(
            {"messages": [], "query": text, "category": "", "answer": ""},
            config={**run_cfg, "run_name": f"m09_kap3_briefing_routing_agent_result_{_i}"},
        )
        tatsächliche_routen.append(result["category"])
        print(
            f"Frage: {text}\n"
            f"Routing: {result['category']}\n"
            f"Antwort: {result['answer']}\n---"
        )

assert tatsächliche_routen == erwartete_routen
print("Routing-Check: alle erwarteten Pfade erreicht")


# 4 | Qualitäts-Gate: Routing mit Schleifen
---

Conditional Routing erlaubt auch **Schleifen** im Graphen:  
Ein Node prüft die Qualität und entscheidet, ob ein weiterer Durchlauf nötig ist.

**Muster: Schreiben → Prüfen → ggf. Nachbessern**

```python
def Qualitäts_router(state: QualityState) -> Literal["nachbessern", "fertig"]:
    if state["quality"] < 0.7 and state["attempts"] < 3:  # Max-Iterations-Schutz!
        return "nachbessern"
    return "fertig"
```

> **Wichtig:** Immer einen **Iterations-Zähler** im State führen und einen  
> Maximalwert prüfen – sonst entstehen Endlosschleifen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Qualitäts-Gate mit Schleife</font> </br></p>
diagram = """%%{init: {'theme':'forest'}}%%
flowchart TD
    START([START]) --> SCHREIB["Schreib-Node"]
    SCHREIB --> CHECK["Prüf-Node\nScore berechnen"]
    CHECK -->|"quality >= 0.7\noder Versuche >= 3"| END([END])
    CHECK -->|"quality < 0.7\nund Versuche < 3"| NACHBES["Nachbesserungs-Node"]
    NACHBES -->|"Versuche + 1"| CHECK
    style CHECK fill:#FF9800,color:#fff
    style NACHBES fill:#9C27B0,color:#fff
"""
mermaid(diagram, width=650)


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Image as IPImage, display

class QualityState(TypedDict):
    topic:    str
    draft:  str
    feedback: str
    quality:    float
    attempts: int

class QualityCheckResult(BaseModel):
    quality:    float = Field(description="Qualitätsscore 0.0-1.0. Ab 0.7 gilt der Text als gut.")
    feedback: str   = Field(description="Konkrete Verbesserungsvorschläge in 1-2 Sätzen.")

check_llm = llm.with_structured_output(QualityCheckResult)

def schreib_node(state: QualityState) -> dict:
    """Schreibt einen Erklärungstext zum Briefing-Thema."""
    basis = f"Erkläre '{state['topic']}' in 3-4 klaren Sätzen für Einsteiger."
    response = llm.invoke([{"role": "user", "content": basis}])
    return {"draft": response.content}

def check_node(state: QualityState) -> dict:
    """Bewertet den Entwurf und gibt Score sowie Feedback zurück."""
    neuer_versuch = state.get("attempts", 0) + 1
    prompt = (
        f"Bewerte diesen Erklärungstext zum Thema '{state['topic']}':\n\n"
        f"{state['draft']}"
    )
    ergebnis = check_llm.invoke([{"role": "user", "content": prompt}])
    mprint(f"Versuch {neuer_versuch} - Score: **{ergebnis.quality:.2f}** | {ergebnis.feedback}")
    return {"quality": ergebnis.quality, "feedback": ergebnis.feedback, "attempts": neuer_versuch}

def nachbesserungs_node(state: QualityState) -> dict:
    """Überarbeitet den Entwurf auf Basis des Feedbacks."""
    prompt = (
        f"Verbessere diesen Text basierend auf dem Feedback.\n\n"
        f"Text: {state['draft']}\n"
        f"Feedback: {state['feedback']}"
    )
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"draft": response.content}

def qualitaets_router(state: QualityState) -> Literal["nachbessern", END]:
    """Schleife solange Score < 0.7 und Versuche < 3."""
    if state["quality"] < 0.7 and state["attempts"] < 3:
        return "nachbessern"
    return END


In [ ]:
quality_builder = StateGraph(QualityState)
quality_builder.add_node("schreiben", schreib_node)
quality_builder.add_node("check", check_node)
quality_builder.add_node("nachbessern", nachbesserungs_node)

quality_builder.add_edge(START, "schreiben")
quality_builder.add_edge("schreiben", "check")
quality_builder.add_conditional_edges(
    "check",
    qualitaets_router,
    {"nachbessern": "nachbessern", END: END},
)
quality_builder.add_edge("nachbessern", "check")

quality_agent = quality_builder.compile()
display(IPImage(quality_agent.get_graph().draw_mermaid_png()))


In [ ]:
from langsmith import tracing_context

run_cfg = {
    "run_name": "m09_kap4_qualitaetsgate",
    "tags": ["m09", "qualitaets-gate", "schleife"],
}

start_state: QualityState = {
    "topic": "Was ist ein KI-Agent?",
    "draft": "",
    "feedback": "",
    "quality": 0.0,
    "attempts": 0,
}

# Ausgewählter Referenzlauf für die Qualitäts-Gate-Demo.
with tracing_context(enabled=True):
    ergebnis = quality_agent.invoke(start_state, config={**run_cfg, "run_name": "m09_kap4_qualitaet_agent_ergebnis"})

mprint("## Qualitäts-Gate — Durchläufe")
mprint(f"**Versuche:** {ergebnis['attempts']}  ")
mprint(f"**Score:** {ergebnis['quality']:.2f}  ")
mprint("")
mprint("**Finaler Text:**")
mprint(ergebnis["draft"])

# Deterministischer Selfcheck der Routing-Logik unabhängig vom LLM-Score.
assert qualitaets_router({"quality": 0.5, "attempts": 1}) == "nachbessern"
assert qualitaets_router({"quality": 0.95, "attempts": 1}) == END
assert qualitaets_router({"quality": 0.4, "attempts": 3}) == END
print("Qualitäts-Gate-Check: Schleifen- und Abbruchbedingungen korrekt")


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Briefing-Routing mit Qualitäts-Gate
</font></p>

Einen StateGraph bauen, der Briefing-Fragen klassifiziert und beantwortet - mit Qualitätsprüfung, Nachbesserungsschleife und Security-Filter.


**Grundlagen**
- Ein Graph klassifiziert Briefing-Fragen und routet zu mindestens 2 Antwort-Nodes.
- Eine Testfrage läuft Ende-zu-Ende durch den Graphen.


In [ ]:
# Grundlagen: BriefingState + Routing-Graph
# Startpunkt: TypedDict + StateGraph

class BriefingState(TypedDict):
    query: str
    category: str
    answer: str
    quality: float
    attempts: int
    blocked: bool

# 1. anfrage_einordnen(): Node, setzt category anhand von query
# 2. briefing_router(): liest category, gibt Node-Namen zurueck
# 3. Mindestens 2 Antwort-Nodes definieren
# 4. StateGraph aufbauen, kompilieren, mit einer Testfrage aufrufen (briefing_agent, grundlagen_result)

**Aufbau**
- Qualitäts-Node und Nachbesserungs-Node ergänzen.
- Nach der Antwort über `qualitaet` routen: `fertig` oder `nachbessern`.
- Mindestens drei Briefing-Fragen testen: Definition, Retrieval, Out-of-Corpus.


In [ ]:
# Aufbau: Qualitäts-Gate mit Schleife
# Startpunkt: briefing_agent aus Grundlagen erweitern

# 1. antwort_qualitaets_check(): Node, bewertet quality, zählt attempts hoch
# 2. antwort_nachbessern(): Node, ergänzt answer bei Bedarf
# 3. qualitaets_route(): Router - bei quality < 0.7 und attempts < 2 "nachbessern", sonst END
# 4. Graph um beide Nodes erweitern, mit drei Testfragen aufrufen (aufbau_ergebnisse)

**Praxis-Transfer: Meeting- & Briefing-Agent**

Anfrage typisieren: Meeting-Briefing, Briefing-Frage mit Quellenbedarf, Follow-up oder unzulässige Frage.

1. Welche reale Arbeitsfrage löst **dieser** Notebook-Baustein im Meeting- & Briefing-Agent?
2. Welche Eingabedaten oder Dokumente braucht er?
3. Welche Ausgabe sollte er liefern?
4. Welche Risiken oder Grenzen bleiben?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?

**Was passiert hier?**

1. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Agent Security](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-security.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
